In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
!pip uninstall transformers -y
!pip install transformers==4.46.3

# **Import Model**


In [3]:
from transformers import (BertTokenizerFast,EncoderDecoderModel)

tokenizer = BertTokenizerFast.from_pretrained('Arashasg/WikiBert2WikiBert')      # local_files_only :: Dont download again
Raw_model = EncoderDecoderModel.from_pretrained('Arashasg/WikiBert2WikiBert')
tokenizer.save_pretrained("/content/gdrive/MyDrive/NLP_Models/Raw_text_summarization")
Raw_model.save_pretrained("/content/gdrive/MyDrive/NLP_Models/Raw_text_summarization")

tokenizer = BertTokenizerFast.from_pretrained("/content/gdrive/MyDrive/NLP_Models/Raw_text_summarization")
Raw_model = EncoderDecoderModel.from_pretrained("/content/gdrive/MyDrive/NLP_Models/Raw_text_summarization")

Config of the encoder: <class 'transformers.models.bert.modeling_bert.BertModel'> is overwritten by shared encoder config: BertConfig {
  "_name_or_path": "HooshvareLab/bert-fa-base-uncased",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "dtype": "float32",
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "return_dict": false,
  "transformers_version": "4.46.3",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 100000
}

Config of the decoder: <class 'transformers.models.bert.modeling_bert.BertLMHeadModel'> is overwritten by shared decoder config: BertConfig {
  "_name_or_path": "Hoos

In [4]:
def generate_summary(text):
    inputs = tokenizer(text ,padding="max_length" ,truncation=True ,max_length=512 ,return_tensors="pt")

    outputs = Raw_model.generate(inputs.input_ids ,attention_mask=inputs.attention_mask,
        max_length=150,
        min_length=10,
        num_beams=2,
        early_stopping=True)

    summary = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return summary[0]

text = '''
هافبک ایرانی تیم فوتبال القطر گفت : در این تیم تمام مسابقات مهم هستند و برای رسیدن به نتیجه مثبت نیاز به تلاش داریم.
'''
summary = generate_summary(text)
print(summary)

هافبک ایرانی تیم فوتبال القطر گفت : برای رسیدن به نتیجه مثبت نیاز به تلاش داریم.


# **Fine-tune**

## dataset

In [9]:
import torch
from transformers import BertTokenizerFast,EncoderDecoderModel,Seq2SeqTrainer,Seq2SeqTrainingArguments,DataCollatorForSeq2Seq
from datasets import Dataset
import numpy as np
import gc
import pandas as pd
pd.set_option('display.max_colwidth', 100)

dataset = pd.read_csv(r'/content/gdrive/MyDrive/Dataset/pn_summary/train.csv',sep='\t')
dataset = dataset[dataset['categories'] == 'ورزش']                                        # only sports
dataset = dataset.rename(columns={'article': 'text'})[['text','summary']]
# dataset = dataset.head(200)
dataset = dataset.dropna(subset=['text', 'summary'])
dataset = dataset[dataset['text'].str.strip() != '']
dataset = dataset[dataset['summary'].str.strip() != '']

def get_length(text):
    return len(tokenizer(text)["input_ids"])
dataset["length"] = dataset["text"].apply(get_length)

dataset = dataset[dataset["length"] <= 800]
dataset


Token indices sequence length is longer than the specified maximum sequence length for this model (837 > 512). Running this sequence through the model will result in indexing errors


,text,summary,length
67,به گزارش ایرنا و پایگاه خبری وزارت ورزش و جوانان، در جلسه‌ای که شب گذشته در محل وزارت ورزش و جوا...,جلسه وزارت ورزش و جوانان با مدیران تیم استقلال برای حل مشکلات این تیم برگزار شد.,133
68,به گزارش ایرنا، دیگو مارادونا دیروز (جمعه) ۶۰ ساله شد. در روزهایی که به دلیل ابتلای یکی از نزدیک...,بازیکن افسانه‌ای فوتبال جهان گفت: برخوردی که باشگاه بارسلونا با لیونل مسی انجام داد، شایسته این ...,237
109,تالین طهماسیان در گفتگو با خبرنگار مهر با یادآوری اینکه قرعه کشی فصل جدید لیگ برتر بسکتبال بانوا...,با تاکید مسئول دپارتمان مسابقات و داوران بانوان فدراسیون بسکتبال، تنها تیم‌هایی در جلسه قرعه‌کشی...,780
152,به گزارش خبرنگار مهر، بعد از مراسم رونمایی از تندیس ویژه پنج مدال آور المپیک و پارالمپیک که امرو...,رئیس سازمان برنامه و بودجه نشست ویژه‌ای با روسای فدراسیون برگزار کرد تا در جریان مشکلات آنها قرا...,143
192,به گزارش خبرنگار مهر، رحمان رضایی پس از باخت دو بر صفر ذوب آهن اصفهان مقابل شهرخودرو مشهد گفت: د...,سرمربی تیم فوتبال ذوب آهن اصفهان با انتقاد تند از مسئولان سازمان لیگ فوتبال ایران گفت: آقایان در...,585
...,...,...,...
81888,به گزارش ایرنا، تیم فوتبال رئال‌مادرید شب گذشته در دیدار خارج از خانه مقابل والنسیا با شکست سنگی...,سرمربی تیم فوتبال رئال‌مادرید، گفت: هیچ بهانه‌ای برای شکست مقابل والنسیا وجود ندارد و من مسوولیت...,218
81952,به گزارش خبرگزاری مهر و به نقل از سایت باشگاه پرسپولیس، تمرین امروز سه‌شنبه پرسپولیس از ساعت ۱۱:...,تیم فوتبال پرسپولیس پس از یک روز تعطیلی، تمریناتش را مجددا از سر گرفته شد.,165
81960,به گزارش ایرنا؛ شنبه شب از هفته پنجم لیگ فوتبال باشگاه‌های آلمان، دربی روهر بین دورتموند و شالکه...,تیم فوتبال بورسیادورتموند در دیدار مقابل شالکه به برتری سه بر صفر رسید تا فاتح دربی روهر شود.,211
81982,سیدمهدی سیدصالحی روز یکشنبه در گفت و گو با ایرنا، اظهار داشت: وقتی که آندره‌آ استراماچونی به است...,مهاجم اسبق استقلال، گفت: آبی‌پوشان محکوم به برد برابر صنعت نفت آبادان هستند زیرا در بازی قبل برا...,625


## Tokenizer

In [ ]:
dataset = Dataset.from_pandas(dataset[['text', 'summary']])                                                         # Conveting pd.dataframe to Hugging-Face dataset

def tokenize_function(examples):                                                                                    # tokenizing text-input

    model_inputs = tokenizer(examples["text"],max_length=512,truncation=True)
    labels = tokenizer(text_target=examples["summary"],max_length=128,truncation=True)         # tokenizing result or summary
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

tokenized_dataset = dataset.map(tokenize_function,batched=True ,remove_columns=dataset.column_names)

## Modeling

In [ ]:
model.to("cuda")

model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.eos_token_id = tokenizer.sep_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

train_test_split = tokenized_dataset.train_test_split(test_size=0.15, seed=42)
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']

print(f"{len(train_dataset)}")
print(f"{len(eval_dataset)}")

## Train

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer,model=model,padding=True)

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/gdrive/MyDrive/NLP_Models/Weights_summarization",
    overwrite_output_dir=True,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    warmup_steps=100,
    weight_decay=0.01,
    save_total_limit=1,               # 1 model save
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,
    fp16=True,                      # GPU
    push_to_hub=False,
    report_to="none",
    load_best_model_at_end=True, 
    metric_for_best_model="eval_loss",  
    greater_is_better=False,  
    dataloader_pin_memory=False, 
    remove_unused_columns=False,)

trainer = Seq2SeqTrainer(model=model,args=training_args,data_collator=data_collator,train_dataset=train_dataset,eval_dataset=eval_dataset,processing_class=tokenizer)

In [ ]:
gc.collect()
trainer.train()

In [ ]:
trainer.save_model("/content/gdrive/MyDrive/NLP_Models/Text_summarization")
tokenizer.save_pretrained("/content/gdrive/MyDrive/NLP_Models/Text_summarization")

# **Prediction**

In [7]:
new_model = EncoderDecoderModel.from_pretrained('/content/gdrive/MyDrive/NLP_Models/Text_summarization')

Config of the encoder: <class 'transformers.models.bert.modeling_bert.BertModel'> is overwritten by shared encoder config: BertConfig {
  "_name_or_path": "HooshvareLab/bert-fa-base-uncased",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "dtype": "float32",
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "return_dict": false,
  "transformers_version": "4.46.3",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 100000
}

Config of the decoder: <class 'transformers.models.bert.modeling_bert.BertLMHeadModel'> is overwritten by shared decoder config: BertConfig {
  "_name_or_path": "Hoos

In [20]:
def generate_summary_1(text):
    inputs = tokenizer(text ,padding="max_length" ,truncation=True ,max_length=512 ,return_tensors="pt")

    outputs = new_model.generate(inputs.input_ids ,attention_mask=inputs.attention_mask,
        max_length=150,
        min_length=10,
        num_beams=2,
        early_stopping=True)

    summary = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return summary[0]

In [35]:
row = 199
text1 = dataset.iloc[row]["text"]
print('Senetence :',dataset.iloc[row]["summary"])

print('Before fine-tune :',generate_summary(text1))
print('After fine-tune  :',generate_summary_1(text1))

Senetence : با اعلام سرمربی تیم فوتبال استقلال کادر فنی این تیم برای فصل آینده تغییر خواهد کرد.
Before fine-tune : سرمربی تیم فوتبال نساجی مازندران از تغییرات در کادر فنی این تیم خبر داد.
After fine-tune  : سرمربی تیم فوتبال استقلال تغییرات صورت گرفته در کادر فنی این تیم را تشریح کرد.
